# Análisis Estadístico — Changeovers DAMM El Prat 2025
## Treball Final MAM-Estadística

**Metodologías:** (1) Goodness of Fit + prior Bayesiano  (2) Tests de permutación para alternativas ordenadas

### Nodo del BCN = Marca × vol × pack × Envase

| Dimensión | Columna | Descripción |
|---|---|---|
| **Marca** | `Marca` | Marca comercial (20 marcas) |
| **vol** | `Tipo Envase` | 1/3, 1/2 o 2/5 |
| **pack** | `Material Precio` | UNI, P12, P24, P6, RETR |
| **Envase** | `Envase` | Código KR##### — referencia de etiqueta/diseño |

| Tipo arista | Qué cambia | Complejidad esperada |
|---|---|---|
| **C_pack** | mismo Marca+vol, distinto pack | Baja |
| **C_brand** | distinta Marca | Media |
| **C0_self** | mismo nodo (reinicio de lote) | Media-alta |
| **C_envase** | mismo Marca+vol+pack, distinto Envase (referencia) | Alta |

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations
from pathlib import Path

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.titlesize': 12, 'axes.labelsize': 11})
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'raw_data').exists() else Path.cwd().parent
BASE = PROJECT_ROOT
print('Imports OK')


---
## 1. Construcción del grafo

In [ ]:
xl = pd.read_excel(BASE / 'raw_data/Cambios 14_17_19_ 2025.xlsx')
hw = pd.read_csv(BASE / 'clean_data/historical_weeks.csv')
xl = xl.merge(hw[['of','line']].drop_duplicates(), left_on='OF', right_on='of', how='left')
xl = xl.dropna(subset=['line'])
xl['line'] = xl['line'].astype(int)

def parse_vol(te):
    for v in ['1/3', '1/2', '2/5']:
        if v in str(te): return v
    return 'UK'

def parse_pack(mp):
    mp = str(mp).upper()
    if any(k in mp for k in ['PACK 24','BANDEJA24','B24']): return 'P24'
    if any(k in mp for k in ['PACK 12','P12']): return 'P12'
    if any(k in mp for k in ['PACK C. 6','PACK 6']): return 'P6'
    if any(k in mp for k in ['RETRACTIL','RETR']): return 'RETR'
    if 'PACK' in mp: return 'PACK'
    return 'UNI'

xl_sorted = xl.sort_values(['line','Fecha Fin']).reset_index(drop=True)
xl_sorted['Marca'] = xl_sorted['Marca'].str.strip()
xl_sorted['vol']   = xl_sorted['Tipo Envase'].apply(parse_vol)
xl_sorted['pack']  = xl_sorted['Material Precio'].apply(parse_pack)
xl_sorted['node']  = (xl_sorted['Marca'] + '|' + xl_sorted['vol'] + '|' +
                      xl_sorted['pack']  + '|' + xl_sorted['Envase'])
xl_sorted['prev_sku'] = xl_sorted.groupby('line')['SKU'].shift(1)
for col in ['Marca','vol','pack','Envase','node']:
    xl_sorted[f'prev_{col}'] = xl_sorted.groupby('line')[col].shift(1)

ch = xl_sorted[
    xl_sorted['Frecuencia Total'].notna() &
    xl_sorted['C. PRINCIPAL'].notna() &
    xl_sorted['prev_sku'].notna()
].copy()
ch = ch.rename(columns={'Frecuencia Total':'hours','C. PRINCIPAL':'ctype_label',
                         'Fecha Fin':'fecha','SKU':'next_sku','Nº de Cambios':'n_cambios'})
ch['fecha'] = pd.to_datetime(ch['fecha'])
ch['edge']  = ch['line'].astype(str) + ':' + ch['prev_node'] + '→' + ch['node']

def classify_edge(r):
    if r['Marca']  != r['prev_Marca']:  return 'C_brand'
    if r['vol']    != r['prev_vol']:    return 'C_vol'
    if r['pack']   != r['prev_pack']:   return 'C_pack'
    if r['Envase'] != r['prev_Envase']: return 'C_envase'
    return 'C0_self'

ch['chtype'] = ch.apply(classify_edge, axis=1)

EDGE_TYPE_ORDER = ['C_pack','C_brand','C_vol','C0_self','C_envase']
GROUP_ORDER  = ['C_pack','C_brand','C0_self','C_envase']
GROUP_LABELS = {
    'C_pack':  'C_pack\n(diff pack)', 'C_brand': 'C_brand\n(diff marca)',
    'C_vol':   'C_vol\n(diff volumen)', 'C0_self': 'C0_self\n(mismo nodo)',
    'C_envase':'C_envase\n(diff envase)',
}

print(f'Changeovers: {len(ch):,}  |  Nodos: {ch["node"].nunique()}  |  Aristas: {ch["edge"].nunique()}')
print(ch['chtype'].value_counts().to_string())


In [ ]:
# Fig 0 — EDA
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(ch['hours'], bins=50, color='steelblue', edgecolor='white', lw=0.4)
axes[0].axvline(ch['hours'].mean(), color='red', ls='--', lw=1.5, label=f'μ={ch["hours"].mean():.2f}h')
axes[0].axvline(ch['hours'].median(), color='orange', ls='--', lw=1.5, label=f'Med={ch["hours"].median():.2f}h')
axes[0].set_xlabel('Horas'); axes[0].set_ylabel('Frecuencia')
axes[0].set_title(f'Distribución global (n={len(ch):,})'); axes[0].legend(fontsize=9)

pal = sns.color_palette('Set2', 4)
data4 = [ch[ch['chtype']==g]['hours'].values for g in GROUP_ORDER]
bp = axes[1].boxplot(data4, patch_artist=True, medianprops={'color':'red','lw':2})
for patch, color in zip(bp['boxes'], pal): patch.set_facecolor(color)
axes[1].set_xticks(range(1,5))
axes[1].set_xticklabels([GROUP_LABELS[g] for g in GROUP_ORDER], fontsize=8)
axes[1].set_ylabel('Horas'); axes[1].set_title('Por tipo de arista')
axes[1].grid(axis='y', alpha=0.3)
for i, arr in enumerate(data4):
    axes[1].text(i+1, arr.mean()+0.1, f'μ={arr.mean():.2f}h', ha='center', fontsize=8, color='navy', fontweight='bold')
plt.suptitle('Fig 0 — Distribución de changeovers', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 2. Método 1 — Goodness of Fit + Prior Bayesiano

**Split temporal:**  H1 = Ene–Jun 2025 (estimación/prior)  |  H2 = Jul–Dic 2025 (validación)

In [ ]:
SPLIT_DATE = pd.Timestamp('2025-07-01')
ch['half'] = ch['fecha'].apply(lambda d: 'H1' if d < SPLIT_DATE else 'H2')
h1 = ch[ch['half']=='H1'].copy()
h2 = ch[ch['half']=='H2'].copy()
print(f'H1 (Ene–Jun): {len(h1):,} obs  |  H2 (Jul–Dic): {len(h2):,} obs')


### 2.1 ¿Qué distribución sigue cada tipo de arista? (sobre H1)

Para cada tipo de arista se ajustan por MLE cuatro familias: **Gamma**, **Weibull**, **LogNormal** e **Inverse Gaussian**. Se comparan mediante AIC y mediante tests de bondad de ajuste:
- **KS** (Kolmogorov-Smirnov): $D = \sup_x |F_n(x) - F(x)|$  
- **CvM** (Cramér-von Mises): $W^2 = \int [F_n(x)-F(x)]^2 dF(x)$

> Con $n$ grande el KS rechaza cualquier distribución paramétrica. El AIC y el estadístico $D$ son los criterios de selección de modelo.

In [ ]:
DIST_CATALOG = {
    'Gamma':    (stats.gamma,    2),
    'Weibull':  (stats.weibull_min, 2),
    'LogNormal':(stats.lognorm,  2),
    'InvGauss': (stats.invgauss, 2),
}

gof_h1_rows = []
best_params = {}   # type → (dist_name, dist_obj, params)

for g in EDGE_TYPE_ORDER:
    s = h1[h1['chtype']==g]['hours'].values
    if len(s) < 10:
        continue
    rows_g = []
    for dname, (dist, k_params) in DIST_CATALOG.items():
        try:
            params = dist.fit(s, floc=0)
            ll  = dist.logpdf(s, *params).sum()
            aic = -2*ll + 2*k_params
            ks_d, ks_p = stats.kstest(s, dist.cdf, args=params)
            cvm_r = stats.cramervonmises(s, dist.cdf, args=params)
            rows_g.append({
                'Tipo': g, 'Dist': dname, 'n': len(s),
                'AIC': round(aic,1), 'KS_D': round(ks_d,4), 'KS_p': round(ks_p,4),
                'CvM_W2': round(cvm_r.statistic,4), 'CvM_p': round(cvm_r.pvalue,4),
                'params': params,
            })
        except Exception:
            pass
    # Best by AIC
    rows_g.sort(key=lambda r: r['AIC'])
    if rows_g:
        best = rows_g[0]
        best_params[g] = (best['Dist'], DIST_CATALOG[best['Dist']][0], best['params'])
    gof_h1_rows.extend(rows_g)

gof_h1 = pd.DataFrame([{k:v for k,v in r.items() if k!='params'} for r in gof_h1_rows])
print('=== Tabla 1 — GoF por tipo de arista en H1 (AIC ↓ = mejor) ===')
print(gof_h1.to_string(index=False))
print()
print('Mejor distribución por tipo (AIC mínimo):')
for g in EDGE_TYPE_ORDER:
    if g in best_params:
        sub = gof_h1[gof_h1['Tipo']==g].sort_values('AIC')
        best_row = sub.iloc[0]
        gamma_row = sub[sub['Dist']=='Gamma'].iloc[0]
        delta = gamma_row['AIC'] - best_row['AIC']
        print(f'  {g:<12} → mejor: {best_row["Dist"]:<10}  '
              f'AIC_best={best_row["AIC"]:.1f}  AIC_Gamma={gamma_row["AIC"]:.1f}  '
              f'ΔAIC(Gamma−best)={delta:+.1f}')


In [ ]:
# Fig 1 — Histogramas H1 por tipo + PDFs de las 4 familias
n_types = len(EDGE_TYPE_ORDER)
fig, axes = plt.subplots(2, n_types, figsize=(4.2*n_types, 7))
if n_types == 1:
    axes = np.array(axes).reshape(2, 1)
colors_d = {'Gamma':'tab:blue','Weibull':'tab:orange','LogNormal':'tab:green','InvGauss':'tab:red'}

for col, g in enumerate(EDGE_TYPE_ORDER):
    s = h1[h1['chtype']==g]['hours'].values
    if len(s) < 5:
        axes[0,col].set_title(f'{g}\nn insuf.'); axes[1,col].axis('off'); continue
    x = np.linspace(0.01, np.percentile(s, 97), 300)

    # Top: histograma + PDFs
    axes[0,col].hist(s, bins=25, density=True, alpha=0.45, color='steelblue')
    for dname, (dist, _) in DIST_CATALOG.items():
        r = next((r for r in gof_h1_rows if r['Tipo']==g and r['Dist']==dname), None)
        if r:
            axes[0,col].plot(x, dist.pdf(x, *r['params']), lw=2, color=colors_d[dname], label=dname)
    axes[0,col].set_xlabel('Horas'); axes[0,col].set_ylabel('Densidad')
    axes[0,col].set_title(f'{GROUP_LABELS[g].replace(chr(10)," ")}\nn={len(s)}', fontsize=9)
    axes[0,col].legend(fontsize=7)

    # Bottom: QQ-plot Gamma
    r_g = next((r for r in gof_h1_rows if r['Tipo']==g and r['Dist']=='Gamma'), None)
    if r_g:
        probs = (np.arange(1, len(s)+1)-.5)/len(s)
        emp_q = np.sort(s)
        th_q  = stats.gamma.ppf(probs, *r_g['params'])
        lim   = max(emp_q.max(), th_q.max())*1.05
        axes[1,col].scatter(th_q, emp_q, s=8, alpha=0.5, color='tab:blue')
        axes[1,col].plot([0,lim],[0,lim],'r--',lw=1.5)
        axes[1,col].set_xlabel('Cuantiles Gamma'); axes[1,col].set_ylabel('Empírico')
        axes[1,col].set_title(f'QQ Gamma (KS_D={r_g["KS_D"]:.3f})', fontsize=9)

plt.suptitle('Fig 1 — Histogramas H1 + 4 familias + QQ-plot Gamma por tipo de arista',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### 2.2 Priors por tipo: MLE según la familia seleccionada en 2.1

Para cada tipo de arista se toma la familia con menor AIC en la sección 2.1 y se ajustan sus parámetros por MLE sobre H1. Es decir, el prior paramétrico no se fuerza a Gamma: si para una categoría ajusta mejor Weibull, LogNormal o InvGauss, esa es la familia que se usa para resumir H1.

La likelihood del bloque bayesiano posterior se mantiene igual; aquí solo se fija la distribución inicial por categoría a partir del resultado empírico de 2.1.

**Prior por tipo de arista** (familia ganadora + MLE sobre H1):


In [ ]:
# MLE en H1 para cada tipo, usando la familia ganadora por AIC en 2.1
type_priors = {}        # type -> {'dist_name', 'dist', 'params', 'mean', ...}
gamma_priors = {}       # se conserva solo como referencia/compatibilidad para secciones Gamma posteriores

def fmt_params(params):
    return '(' + ', '.join(f'{p:.3f}' for p in params) + ')'

def fit_catalog_for_sample(s, tipo):
    rows = []
    for dname, (dist, k_params) in DIST_CATALOG.items():
        try:
            params = dist.fit(s, floc=0)
            ll = dist.logpdf(s, *params).sum()
            aic = -2*ll + 2*k_params
            ks_d, ks_p = stats.kstest(s, dist.cdf, args=params)
            cvm_r = stats.cramervonmises(s, dist.cdf, args=params)
            rows.append({
                'Tipo': tipo, 'Dist': dname, 'n': len(s), 'AIC': aic,
                'KS_D': ks_d, 'KS_p': ks_p, 'CvM_W2': cvm_r.statistic,
                'CvM_p': cvm_r.pvalue, 'params': params,
            })
        except Exception:
            pass
    return sorted(rows, key=lambda r: r['AIC'])

print('=== Prior por tipo de arista: familia seleccionada por AIC + MLE en H1 ===')
print(f'{"Tipo":<12}  {"n_H1":>5}  {"Familia":<10}  {"μ_prior":>8}  {"AIC":>9}  {"KS_p":>7}  {"CvM_p":>7}  params')

for g in EDGE_TYPE_ORDER:
    s = h1[h1['chtype']==g]['hours'].values
    if len(s) < 5:
        print(f'{g:<12}  n insuficiente')
        continue

    candidates = [r for r in gof_h1_rows if r['Tipo'] == g]
    candidates = sorted(candidates, key=lambda r: r['AIC'])
    best = candidates[0]
    dist = DIST_CATALOG[best['Dist']][0]
    params = best['params']
    mu = dist.mean(*params)
    type_priors[g] = {
        'dist_name': best['Dist'], 'dist': dist, 'params': params, 'mean': mu,
        'n_h1': len(s), 'AIC': best['AIC'], 'KS_p': best['KS_p'], 'CvM_p': best['CvM_p'],
    }

    # Referencia Gamma, no decisión metodológica principal.
    a, loc, sc = stats.gamma.fit(s, floc=0)
    gamma_priors[g] = (a, loc, sc)

    print(f'{g:<12}  {len(s):>5}  {best["Dist"]:<10}  {mu:>8.3f}h  '
          f'{best["AIC"]:>9.1f}  {best["KS_p"]:>7.4f}  {best["CvM_p"]:>7.4f}  {fmt_params(params)}')

# Prior global: misma regla, usando todas las observaciones H1.
global_candidates = fit_catalog_for_sample(h1['hours'].values, 'GLOBAL')
global_best = global_candidates[0]
global_dist = DIST_CATALOG[global_best['Dist']][0]
global_params = global_best['params']
global_mu = global_dist.mean(*global_params)
type_priors['GLOBAL'] = {
    'dist_name': global_best['Dist'], 'dist': global_dist, 'params': global_params,
    'mean': global_mu, 'n_h1': len(h1), 'AIC': global_best['AIC'],
    'KS_p': global_best['KS_p'], 'CvM_p': global_best['CvM_p'],
}

# Parámetros Gamma globales conservados para el bloque de likelihood/posterior y el fallback Empirical Bayes.
a_g, loc_g, sc_g = stats.gamma.fit(h1['hours'].values, floc=0)
gamma_priors['GLOBAL'] = (a_g, loc_g, sc_g)

print(f'{"GLOBAL":<12}  {len(h1):>5}  {global_best["Dist"]:<10}  {global_mu:>8.3f}h  '
      f'{global_best["AIC"]:>9.1f}  {global_best["KS_p"]:>7.4f}  {global_best["CvM_p"]:>7.4f}  {fmt_params(global_params)}')


### 2.3 Actualización secuencial del posterior: prior informativa vs no informativa

**Modelo conjugado Gamma-Exponencial:**

Mantenemos la misma likelihood para las nuevas observaciones:

$$X_i \mid \lambda \sim \text{Exp}(\lambda), \qquad \lambda \sim \text{Gamma}(a,b)$$

donde $b$ es la tasa del prior Gamma sobre $\lambda$.

| Caso | Inicialización | Interpretación |
|---|---|---|
| **Prior informativa** | $a_0=n_{H1}$, $b_0=\sum_{H1}x_i$ | H1 entra como pseudo-muestra histórica. |
| **Prior uniforme/no informativa** | $p(\lambda)\propto 1$  →  $a_0=1$, $b_0=0$ | No aporta escala histórica; aprende solo de H2. |
| **Posterior tras H2** | $\lambda\mid\mathbf{x}\sim\text{Gamma}(a_0+n,\;b_0+\sum x_i)$ | Misma likelihood, distinto punto de partida. |
| **Duración esperada** | $E[X]=E[1/\lambda]=\dfrac{b}{a-1}$ | Definida para $a>1$. |

La pregunta práctica es si, al incorporar suficientes observaciones H2, la posterior informativa y la no informativa terminan dando una duración esperada parecida.


In [ ]:
def sequential_bayes(h1_obs, h2_obs, prior='informative'):
    """Gamma-Exponential conjugate sequential update.

    informative:  λ ~ Gamma(a0=n_H1, rate=b0=sum_H1)
    uniform:      p(λ) ∝ 1, equivalent to Gamma(a0=1, rate=b0=0)

    Posterior after each H2 observation: Gamma(a0+k, b0+sum_H2[:k])
    E[X] = b/(a-1)  (posterior predictive mean, when a > 1)
    95% CI for E[X]: inverse transform of Gamma quantiles for λ.
    """
    h1_obs = np.asarray(h1_obs, dtype=float)
    h2_obs = np.asarray(h2_obs, dtype=float)

    if prior == 'informative':
        a0 = len(h1_obs)
        b0 = h1_obs.sum()
    elif prior == 'uniform':
        a0 = 1.0
        b0 = 0.0
    else:
        raise ValueError("prior must be 'informative' or 'uniform'")

    a_seq, b_seq = [a0], [b0]
    for x in h2_obs:
        a_seq.append(a_seq[-1] + 1)
        b_seq.append(b_seq[-1] + x)

    a_arr = np.array(a_seq, dtype=float)
    b_arr = np.array(b_seq, dtype=float)
    mean_seq = np.where(a_arr > 1, b_arr / (a_arr - 1), np.nan)

    ci_lo, ci_hi = [], []
    for a, b in zip(a_arr, b_arr):
        if a <= 1 or b <= 0:
            ci_lo.append(np.nan); ci_hi.append(np.nan)
            continue
        lam_hi = stats.gamma.ppf(0.975, a, scale=1/b)
        lam_lo = stats.gamma.ppf(0.025, a, scale=1/b)
        ci_lo.append(1 / lam_hi)
        ci_hi.append(1 / lam_lo)

    return a_arr, b_arr, mean_seq, np.array(ci_lo), np.array(ci_hi)

print('Función sequential_bayes definida.')
print('Muestra: prior informativa vs uniforme en las primeras 5 observaciones H2:')
for g in ['C_brand', 'C_envase']:
    s_h1 = h1[h1['chtype'] == g]['hours'].values
    s_h2 = h2[h2['chtype'] == g].sort_values('fecha')['hours'].values
    if len(s_h1) < 2 or len(s_h2) < 2:
        continue

    print(f'\n  {g} | n_H1={len(s_h1)}, n_H2={len(s_h2)}')
    for prior_name in ['informative', 'uniform']:
        a_arr, b_arr, mean_seq, ci_lo, ci_hi = sequential_bayes(s_h1, s_h2[:5], prior=prior_name)
        print(f'  Prior {prior_name}: a0={a_arr[0]:.0f}, b0={b_arr[0]:.1f}, E0={mean_seq[0] if np.isfinite(mean_seq[0]) else np.nan:.3f}h')
        for k in range(1, len(a_arr)):
            xi = s_h2[k-1]
            print(f'    Obs H2 #{k} (x={xi:.2f}h): E[X]={mean_seq[k]:.3f}h  '
                  f'95%CI=[{ci_lo[k]:.3f}, {ci_hi[k]:.3f}]')

# Tabla de convergencia final por tipo.
conv_rows = []
for g in EDGE_TYPE_ORDER:
    s_h1 = h1[h1['chtype'] == g]['hours'].values
    s_h2 = h2[h2['chtype'] == g].sort_values('fecha')['hours'].values
    if len(s_h1) < 2 or len(s_h2) < 2:
        continue
    _, _, mean_inf, _, _ = sequential_bayes(s_h1, s_h2, prior='informative')
    _, _, mean_uni, _, _ = sequential_bayes(s_h1, s_h2, prior='uniform')
    diff = mean_inf[-1] - mean_uni[-1]
    rel = 100 * diff / mean_uni[-1] if mean_uni[-1] else np.nan
    conv_rows.append({
        'Tipo': g, 'n_H1': len(s_h1), 'n_H2': len(s_h2),
        'Posterior inf. final': round(mean_inf[-1], 3),
        'Posterior uniform final': round(mean_uni[-1], 3),
        'Dif. abs. (h)': round(diff, 3), 'Dif. rel. (%)': round(rel, 2),
    })

posterior_convergence = pd.DataFrame(conv_rows)
print('\n=== Convergencia final: prior informativa vs uniforme ===')
print(posterior_convergence.to_string(index=False))


In [ ]:
# Fig 2 — Convergencia del posterior: prior informativa vs uniforme para todos los tipos
plot_types = [g for g in EDGE_TYPE_ORDER
              if len(h1[h1['chtype'] == g]) >= 2 and len(h2[h2['chtype'] == g]) >= 2]

fig = plt.figure(figsize=(16, 8.4))
gs = fig.add_gridspec(2, 6, hspace=0.38, wspace=0.55)
positions = [(0, 0, 2), (0, 2, 4), (0, 4, 6), (1, 1, 3), (1, 3, 5)]
axes = [fig.add_subplot(gs[row, c0:c1]) for row, c0, c1 in positions[:len(plot_types)]]
colors = sns.color_palette('Set2', len(plot_types))

for ax, g, color in zip(axes, plot_types, colors):
    s_h1 = h1[h1['chtype'] == g]['hours'].values
    s_h2 = h2[h2['chtype'] == g].sort_values('fecha')['hours'].values

    a_i, b_i, mean_i, lo_i, hi_i = sequential_bayes(s_h1, s_h2, prior='informative')
    a_u, b_u, mean_u, lo_u, hi_u = sequential_bayes(s_h1, s_h2, prior='uniform')
    k_vals = np.arange(len(mean_i))

    ax.fill_between(k_vals, lo_i, hi_i, alpha=0.18, color=color, label='95% CI informative')
    ax.fill_between(k_vals, lo_u, hi_u, alpha=0.10, color='gray', label='95% CI weak')
    ax.plot(k_vals, mean_i, '-', color=color, lw=2.2, label='Informative')
    ax.plot(k_vals, mean_u, '--', color='black', lw=1.8, label='Weak')
    ax.axhline(s_h1.mean(), color=color, ls=':', lw=1.2, alpha=0.8, label=f'H1 mean={s_h1.mean():.2f}h')
    ax.axhline(s_h2.mean(), color='black', ls=':', lw=1.2, alpha=0.8, label=f'H2 mean={s_h2.mean():.2f}h')

    diff_final = mean_i[-1] - mean_u[-1]
    ax.scatter([k_vals[-1]], [mean_i[-1]], color=color, s=30, zorder=5)
    ax.scatter([k_vals[-1]], [mean_u[-1]], color='black', s=30, zorder=5)
    ax.text(k_vals[-1], max(mean_i[-1], mean_u[-1]),
            f'  Δ={diff_final:+.2f}h', va='bottom', fontsize=8)

    ax.set_xlabel('New H2 observations')
    ax.set_ylabel(f'{g}\nPosterior E[X] (h)')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.show()


In [ ]:
# Fig 3 — Antes vs después: distribuciones posteriores con prior informativa y uniforme

def expected_duration_pdf(mu_grid, a, b):
    """Density of μ = 1/λ when λ ~ Gamma(a, rate=b)."""
    mu_grid = np.asarray(mu_grid, dtype=float)
    if a <= 0 or b <= 0:
        return np.full_like(mu_grid, np.nan, dtype=float)
    lam = 1 / mu_grid
    return stats.gamma.pdf(lam, a, scale=1/b) / (mu_grid ** 2)

plot_types = [g for g in EDGE_TYPE_ORDER
              if len(h1[h1['chtype'] == g]) >= 2 and len(h2[h2['chtype'] == g]) >= 2]

n_types = len(plot_types)
fig, axes = plt.subplots(n_types, 3, figsize=(14, 3.1*n_types), sharex=False, sharey=False)
if n_types == 1:
    axes = axes.reshape(1, 3)

for row, g in enumerate(plot_types):
    ax_before, ax_after, ax_shift = axes[row]
    s_h1 = h1[h1['chtype'] == g]['hours'].values
    s_h2 = h2[h2['chtype'] == g].sort_values('fecha')['hours'].values

    a_i, b_i, mean_i, _, _ = sequential_bayes(s_h1, s_h2, prior='informative')
    a_u, b_u, mean_u, _, _ = sequential_bayes(s_h1, s_h2, prior='uniform')
    k_end = len(s_h2)

    x_max = np.nanpercentile(np.concatenate([s_h1, s_h2]), 98) * 2.4
    x_max = max(x_max, np.nanmax([mean_i[0], mean_i[k_end], mean_u[k_end]]) * 2.1)
    mu_grid = np.linspace(0.03, x_max, 700)

    pdf_i0 = expected_duration_pdf(mu_grid, a_i[0], b_i[0])
    pdf_i_end = expected_duration_pdf(mu_grid, a_i[k_end], b_i[k_end])
    pdf_u_end = expected_duration_pdf(mu_grid, a_u[k_end], b_u[k_end])

    # ANTES: informative prior is proper; uniform prior is conceptual/improper before seeing H2.
    ax_before.plot(mu_grid, pdf_i0, color='tab:blue', lw=2.4, label='Informativa: prior H1')
    ax_before.axvline(mean_i[0], color='tab:blue', ls='-', lw=1.5, alpha=0.8)
    ax_before.axhline(0, color='black', lw=2.0, alpha=0.55, label='Uniforme: p(λ) ∝ 1')
    ax_before.text(0.05*x_max, max(np.nanmax(pdf_i0), 1e-9)*0.88,
                   'Uniforme inicial\nno normalizable', fontsize=8, color='dimgray', va='top')
    ax_before.set_title('ANTES de H2', fontsize=10, fontweight='bold')
    ax_before.set_ylabel(f'{g}\nDensidad')
    ax_before.set_xlim(0, x_max)
    ax_before.grid(alpha=0.25)

    # DESPUÉS: both are proper posteriors after all H2 observations.
    ax_after.plot(mu_grid, pdf_i_end, color='tab:blue', lw=2.5,
                  label=f'Informativa final ({mean_i[k_end]:.2f}h)')
    ax_after.plot(mu_grid, pdf_u_end, color='black', lw=2.5, ls='--',
                  label=f'Uniforme final ({mean_u[k_end]:.2f}h)')
    ax_after.axvline(s_h2.mean(), color='gray', ls=':', lw=1.5, label=f'Media H2={s_h2.mean():.2f}h')
    ax_after.set_title(f'DESPUÉS de H2 (k={k_end})', fontsize=10, fontweight='bold')
    ax_after.set_xlim(0, x_max)
    ax_after.grid(alpha=0.25)

    # SHIFT: compact visual summary of start/end posterior means.
    y_inf, y_uni = 1.0, 0.35
    ax_shift.plot([0, 1], [y_inf, y_inf], color='tab:blue', lw=2)
    ax_shift.scatter([0, 1], [y_inf, y_inf], s=55, color='tab:blue', zorder=3)
    ax_shift.text(0, y_inf + 0.08, f'{mean_i[0]:.2f}h', ha='center', fontsize=8)
    ax_shift.text(1, y_inf + 0.08, f'{mean_i[k_end]:.2f}h', ha='center', fontsize=8)
    ax_shift.text(-0.08, y_inf, 'Informativa', ha='right', va='center', fontsize=9, color='tab:blue')

    ax_shift.plot([0, 1], [y_uni, y_uni], color='black', lw=2, ls='--')
    ax_shift.scatter([1], [y_uni], s=55, color='black', zorder=3)
    ax_shift.text(0, y_uni + 0.08, 'impropia', ha='center', fontsize=8, color='dimgray')
    ax_shift.text(1, y_uni + 0.08, f'{mean_u[k_end]:.2f}h', ha='center', fontsize=8)
    ax_shift.text(-0.08, y_uni, 'Uniforme', ha='right', va='center', fontsize=9, color='black')
    ax_shift.axvline(0, color='gray', lw=1, alpha=0.5)
    ax_shift.axvline(1, color='gray', lw=1, alpha=0.5)
    ax_shift.set_xlim(-0.35, 1.15); ax_shift.set_ylim(0, 1.35)
    ax_shift.set_xticks([0, 1]); ax_shift.set_xticklabels(['antes', 'después'])
    ax_shift.set_yticks([])
    ax_shift.set_title(f'Δ final={mean_i[k_end]-mean_u[k_end]:+.2f}h', fontsize=10, fontweight='bold')
    ax_shift.grid(axis='x', alpha=0.2)

    if row == n_types - 1:
        ax_before.set_xlabel('Duración esperada μ = 1/λ (h)')
        ax_after.set_xlabel('Duración esperada μ = 1/λ (h)')

    ax_before.legend(fontsize=8, loc='upper right')
    ax_after.legend(fontsize=8, loc='upper right')

plt.suptitle('Fig 3 — Antes vs después: prior uniforme/no informativa e informativa tras observar H2',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


### 2.4 Tests de comparación H1 vs H2 (KS, Anderson-Darling, Cramér-von Mises)

¿Son las distribuciones de H1 y H2 estadísticamente iguales? Si el proceso es estacionario, los parámetros del prior siguen siendo válidos en H2.

| Test | $H_0$ | Estadístico |
|---|---|---|
| **KS** (Kolmogorov-Smirnov) | $F_{H1}=F_{H2}$ | $\sup_x|F_{n1}(x)-F_{n2}(x)|$ |
| **AD** (Anderson-Darling) | $F_{H1}=F_{H2}$ | Ponderado en las colas |
| **CvM** (Cramér-von Mises) | $F_{H1}=F_{H2}$ | $\int[F_{n1}-F_{n2}]^2$ |

In [ ]:
try:
    from scipy.stats import cramervonmises_2samp as cvm2
    HAS_CVM2 = True
except ImportError:
    HAS_CVM2 = False
    print('cramervonmises_2samp no disponible (scipy < 1.7). Se omite CvM two-sample.')

compare_rows = []
groups_to_test = EDGE_TYPE_ORDER + ['GLOBAL']

for g in groups_to_test:
    s1 = h1['hours'].values if g=='GLOBAL' else h1[h1['chtype']==g]['hours'].values
    s2 = h2['hours'].values if g=='GLOBAL' else h2[h2['chtype']==g]['hours'].values
    if len(s1)<5 or len(s2)<5:
        continue

    ks_d, ks_p = stats.ks_2samp(s1, s2)
    ad_r = stats.anderson_ksamp([s1, s2])
    cvm_stat, cvm_p = (None, None)
    if HAS_CVM2:
        cvm_r = cvm2(s1, s2)
        cvm_stat, cvm_p = cvm_r.statistic, cvm_r.pvalue

    row = {
        'Tipo': g, 'n_H1': len(s1), 'n_H2': len(s2),
        'KS_D': round(ks_d,4), 'KS_p': round(ks_p,4),
        'AD_stat': round(ad_r.statistic,3),
        'AD_sig_lv': ad_r.significance_level,
    }
    if cvm_stat is not None:
        row['CvM_stat'] = round(cvm_stat,4)
        row['CvM_p']    = round(cvm_p,4)
    compare_rows.append(row)

tab_compare = pd.DataFrame(compare_rows)
print('=== Tabla 2 — Tests de comparación H1 vs H2 (two-sample) ===')
print(tab_compare.to_string(index=False))
print()
print('KS_p > 0.05 → no se rechaza H₀ (H1 y H2 provienen de la misma distribución).')
print('AD significance_level: nivel al que se rechaza (25/15/10/5/1%).')
print('CvM_p > 0.05 → no se rechaza H₀.')
print()
print('Interpretación: si no se rechaza H₀, el prior estimado en H1 sigue siendo válido en H2.')


### 2.5 Fase B — GoF de la familia seleccionada por tipo (H1 estima → H2 valida)

Una vez seleccionada la familia por AIC en H1 para cada tipo de arista, comprobamos si ese ajuste MLE predice razonablemente H2. La comparación H1→H2 usa la misma familia elegida en 2.1 para cada categoría.


In [ ]:
# GoF H1 -> H2 usando la familia seleccionada en 2.1 para cada tipo
validation_rows = []

def validation_aic(dist, params, s2, k_params):
    ll = dist.logpdf(s2, *params).sum()
    return -2*ll + 2*k_params

for g in EDGE_TYPE_ORDER + ['GLOBAL']:
    s2 = h2['hours'].values if g == 'GLOBAL' else h2[h2['chtype'] == g]['hours'].values
    prior = type_priors.get(g)
    if prior is None or len(s2) < 5:
        validation_rows.append({'Tipo': g, 'n_H2': len(s2), 'Familia': '—', 'AIC_H2': '—', 'KS_D': '—', 'KS_p': '—', 'Decisión': 'n insuf.'})
        continue

    dist_name = prior['dist_name']
    dist = prior['dist']
    params = prior['params']
    k_params = DIST_CATALOG[dist_name][1]
    aic_h2 = validation_aic(dist, params, s2, k_params)
    ks_d, ks_p = stats.kstest(s2, dist.cdf, args=params)
    validation_rows.append({
        'Tipo': g, 'n_H1': prior['n_h1'], 'n_H2': len(s2), 'Familia': dist_name,
        'μ_H1': round(prior['mean'], 3), 'AIC_H2': round(aic_h2, 1),
        'KS_D': round(ks_d, 4), 'KS_p': round(ks_p, 4),
        'Decisión': 'compatible' if ks_p > 0.05 else 'revisar shift',
    })

tab_validation = pd.DataFrame(validation_rows)
print('=== Tabla 3 — Validación H1→H2 con la familia seleccionada en 2.1 ===')
print(tab_validation.to_string(index=False))

selected_global_name = type_priors['GLOBAL']['dist_name']
selected_global_aic_h2 = tab_validation.loc[tab_validation['Tipo'] == 'GLOBAL', 'AIC_H2'].iloc[0]
selected_global_ks_p = tab_validation.loc[tab_validation['Tipo'] == 'GLOBAL', 'KS_p'].iloc[0]
print()
print(f'Global seleccionado: {selected_global_name} | AIC_H2={selected_global_aic_h2} | KS_p={selected_global_ks_p}')


In [ ]:
# Fig 4 — Histogramas H1/H2 + familia seleccionada por tipo, sin GLOBAL
plot_types = EDGE_TYPE_ORDER
fig = plt.figure(figsize=(16, 8.4))
gs = fig.add_gridspec(2, 6, hspace=0.38, wspace=0.55)
positions = [(0, 0, 2), (0, 2, 4), (0, 4, 6), (1, 1, 3), (1, 3, 5)]
axes = [fig.add_subplot(gs[row, c0:c1]) for row, c0, c1 in positions[:len(plot_types)]]
pal = sns.color_palette('Set2', len(plot_types))

for ax, g, color in zip(axes, plot_types, pal):
    s1 = h1[h1['chtype'] == g]['hours'].values
    s2 = h2[h2['chtype'] == g]['hours'].values
    prior = type_priors.get(g)
    if prior is None or len(s1) < 5:
        ax.axis('off')
        continue
    dist = prior['dist']
    params = prior['params']
    x = np.linspace(0.01, np.percentile(np.concatenate([s1, s2]), 97), 300)
    ax.hist(s1, bins=25, density=True, alpha=0.50, color=color, label=f'H1 (n={len(s1)})')
    ax.hist(s2, bins=25, density=True, alpha=0.30, color='gray', label=f'H2 (n={len(s2)})')
    ax.plot(x, dist.pdf(x, *params), '-', color=color, lw=2.4, label=prior['dist_name'])
    ax.set_xlabel('Hours')
    ax.set_ylabel(f'{g}\nDensity')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.25)

plt.show()


### 2.6 Prior específico por arista (Empirical Bayes)

Para aristas con $n \geq 2$ observaciones en H1, se estiman $\hat{\alpha}_{ij}, \hat{s}_{ij}$ individualmente. El resto usa el prior global como regularizador jerárquico.

In [ ]:
MIN_OBS = 2
edge_records = []
for edge, grp in ch.groupby('edge'):
    obs = grp['hours'].values; n = len(obs)
    if n >= MIN_OBS:
        try:
            a_e,_,sc_e = stats.gamma.fit(obs, floc=0); source='edge_specific'
        except:
            a_e,sc_e,source = a_g,sc_g,'global_fallback'
    else:
        a_e,sc_e,source = a_g,sc_g,'global_fallback'
    edge_records.append({
        'edge':edge,'line':grp['line'].iloc[0],
        'prev_node':grp['prev_node'].iloc[0],'next_node':grp['node'].iloc[0],
        'chtype':grp['chtype'].iloc[0],'n_obs':n,
        'alpha':round(a_e,4),'scale':round(sc_e,4),
        'mean_prior':round(a_e*sc_e,3),'prior_source':source,
    })

edge_priors = pd.DataFrame(edge_records)
n_specific = (edge_priors['prior_source']=='edge_specific').sum()
n_global   = (edge_priors['prior_source']=='global_fallback').sum()

print(f'Aristas totales: {len(edge_priors):,}')
print(f'  Prior específico (n≥{MIN_OBS}): {n_specific} ({100*n_specific/len(edge_priors):.1f}%)')
print(f'  Prior global     (n<{MIN_OBS}):  {n_global} ({100*n_global/len(edge_priors):.1f}%)')
print(f'\nPrior global: Gamma(α={a_g:.3f}, scale={sc_g:.3f}) → μ={a_g*sc_g:.2f}h')


In [ ]:
# Fig 5 — Distribución de α y scatter (α, scale)
ep_sp = edge_priors[edge_priors['prior_source']=='edge_specific'].copy()
pct_gt1 = 100*(ep_sp['alpha']>1).mean()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(ep_sp['alpha'].clip(upper=20), bins=30, color='steelblue', edgecolor='white', lw=0.4)
axes[0].axvline(1.0, color='red', ls='--', lw=2, label='α=1 (Exponencial)')
axes[0].axvline(ep_sp['alpha'].median(), color='orange', ls='--', lw=1.5,
                label=f'Mediana α={ep_sp["alpha"].median():.2f}')
axes[0].set_xlabel('α (cap=20)'); axes[0].set_ylabel('Aristas')
axes[0].set_title(f'Fig 5a — α por arista (n={len(ep_sp)}, {pct_gt1:.0f}% con α>1)')
axes[0].legend(fontsize=9)

colors_l = {14:'tab:blue',17:'tab:orange',19:'tab:green'}
for lid, grp in ep_sp.groupby('line'):
    axes[1].scatter(grp['alpha'].clip(upper=20), grp['scale'],
                    s=20, alpha=0.6, label=f'L{lid}', color=colors_l.get(lid,'gray'))
axes[1].axvline(1.0, color='red', ls='--', lw=1.5, alpha=0.6)
axes[1].set_xlabel('α (cap=20)'); axes[1].set_ylabel('scale')
axes[1].set_title('Fig 5b — (α, scale) por línea'); axes[1].legend(fontsize=9)

plt.suptitle(f'Fig 5 — Parámetros Gamma por arista  |  α>1 en {pct_gt1:.0f}% → setup físico confirmado',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


**Conclusión Método 1:**
- La familia usada como prior paramétrico se selecciona por tipo de arista según el AIC de la sección 2.1 y se ajusta por MLE en H1.  
- Gamma no se impone cuando otra familia ajusta mejor; queda solo como referencia y como familia del bloque conjugado donde se mantiene la likelihood definida.  
- Los tests H1 vs H2 permiten verificar estacionaridad: si $p>0.05$, el prior estimado en H1 sigue siendo válido en H2.  
- La proporción de aristas con $\alpha>1$ se calcula en la figura de priors específicos Gamma; se interpreta solo dentro de ese bloque de Empirical Bayes.



---
## 3. Método 2 — Tests de Permutación para Alternativas Ordenadas

El test principal ya es un contraste de igualdad frente a una alternativa ordenada.
Por tanto, no usamos un omnibus separado antes del contraste ordenado: bajo
\(H_0\), las etiquetas de grupo son intercambiables; bajo \(H_A\), los rangos medios
siguen el orden esperado.

Para \(k\) grupos ordenados se calculan los rangos medios conjuntos \(R_i\) y se usa
la suma de todas las diferencias ordenadas:

$$
T_k=\sum_{i<j}(R_j-R_i).
$$

Para \(k=4\), esto equivale exactamente al estadístico de la hoja:

$$
T_4 = 3R_4 + R_3 - R_2 - 3R_1.
$$

Para \(k=3\), que es el caso de las líneas, se obtiene:

$$
T_3 = (R_2-R_1)+(R_3-R_1)+(R_3-R_2)=2R_3-2R_1.
$$

Valores grandes de \(T_k\) apoyan la alternativa ordenada. La distribución nula se
calibra con 10.000 permutaciones de etiquetas.


In [ ]:

def rank_means(groups):
    combined = np.concatenate(groups)
    ranks = stats.rankdata(combined)
    R, idx = [], 0
    for g in groups:
        R.append(ranks[idx:idx+len(g)].mean())
        idx += len(g)
    return np.array(R)

def ordered_rank_statistic(groups):
    """T_k = sum_{i<j}(R_j - R_i). Large values support the ordered alternative."""
    R = rank_means(groups)
    return sum(R[j] - R[i] for i in range(len(R)) for j in range(i+1, len(R)))

def pair_order_statistic(groups):
    """Two-group ordered rank statistic: large values support group 1 < group 2."""
    R1, R2 = rank_means(groups)
    return R2 - R1

def permutation_test(groups, stat_fn, n_perm=10_000, seed=42, alternative='greater'):
    rng = np.random.default_rng(seed)
    sizes = [len(g) for g in groups]
    combined = np.concatenate(groups)
    T_obs = stat_fn(groups)
    T_null = []
    for _ in range(n_perm):
        p = rng.permutation(combined)
        pg, idx = [], 0
        for n_i in sizes:
            pg.append(p[idx:idx+n_i])
            idx += n_i
        T_null.append(stat_fn(pg))
    T_null = np.array(T_null)
    if alternative == 'greater':
        p_val = (np.sum(T_null >= T_obs) + 1) / (len(T_null) + 1)
    elif alternative == 'two-sided':
        center = np.mean(T_null)
        p_val = (np.sum(np.abs(T_null - center) >= abs(T_obs - center)) + 1) / (len(T_null) + 1)
    else:
        raise ValueError("alternative must be 'greater' or 'two-sided'")
    return T_obs, T_null, p_val

print('Funciones definidas: rank_means, ordered_rank_statistic, pair_order_statistic, permutation_test')



### 3.1 Aplicación A — Por tipo de arista (k=4)

Queremos contrastar:

$$
H_0:m_1=m_2=m_3=m_4
$$

frente a la alternativa ordenada:

$$
H_A:m_{C\_pack}\leq m_{C\_brand}\leq m_{C0\_self}\leq m_{C\_envase},
$$

con al menos una desigualdad estricta. Como las duraciones son asimétricas y los
tamaños muestrales están desbalanceados, usamos rangos en lugar de ANOVA.

El estadístico principal es:

$$
T_4 = 3R_4 + R_3 - R_2 - 3R_1,
$$

que es la suma de todas las diferencias ordenadas \(R_j-R_i\). Bajo \(H_0\), las
etiquetas de los cuatro grupos son intercambiables, así que obtenemos el
\(p\)-valor permutando etiquetas.


In [ ]:

groups_type = [ch[ch['chtype'] == g]['hours'].values for g in GROUP_ORDER]
R_type = rank_means(groups_type)

print('=== Tabla 4 — Estadísticos por tipo de arista ===')
tab4 = pd.DataFrame([
    {'Tipo': g, 'n': len(arr), 'Media (h)': round(arr.mean(), 3),
     'Mediana (h)': round(np.median(arr), 3), 'Rango medio Rᵢ': round(ri, 1)}
    for g, arr, ri in zip(GROUP_ORDER, groups_type, R_type)
])
print(tab4.to_string(index=False))

T_obs_type, T_null_type, p_val_type = permutation_test(
    groups_type, ordered_rank_statistic, n_perm=10_000, seed=43, alternative='greater'
)
print(f'\nTest ordenado por permutación: T_obs={T_obs_type:.2f}, p={p_val_type:.4f}')
if p_val_type < 0.05:
    print('→ Rechazamos H₀: hay evidencia a favor del orden C_pack ≤ C_brand ≤ C0_self ≤ C_envase.')


In [ ]:

# Fig 6 — Boxplots, rangos medios y distribución nula del estadístico ordenado
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
pal = sns.color_palette('Set2', 4)

bp = axes[0].boxplot(groups_type, patch_artist=True, medianprops={'color': 'red', 'lw': 2})
for patch, color in zip(bp['boxes'], pal):
    patch.set_facecolor(color)
axes[0].set_xticks(range(1, 5))
axes[0].set_xticklabels([GROUP_LABELS[g] for g in GROUP_ORDER], fontsize=8)
axes[0].set_ylabel('Horas')
axes[0].set_title('Fig 6a — Boxplots por tipo')
axes[0].grid(axis='y', alpha=0.3)
for i, arr in enumerate(groups_type):
    axes[0].text(i+1, arr.mean()+0.1, f'μ={arr.mean():.2f}h',
                 ha='center', fontsize=8, color='navy', fontweight='bold')

axes[1].bar(range(1, 5), R_type, color=pal, edgecolor='black', lw=0.6)
axes[1].set_xticks(range(1, 5))
axes[1].set_xticklabels([GROUP_LABELS[g] for g in GROUP_ORDER], fontsize=8)
axes[1].set_ylabel('Rango medio Rᵢ')
axes[1].set_title('Fig 6b — Rangos medios')
axes[1].grid(axis='y', alpha=0.3)
for i, r in enumerate(R_type):
    axes[1].text(i+1, r+4, f'R={r:.0f}', ha='center', fontsize=9)

axes[2].hist(T_null_type, bins=60, density=True, color='lightsteelblue', edgecolor='white', lw=0.4,
             label='Nula: etiquetas permutadas')
axes[2].axvline(T_obs_type, color='red', lw=2.5, label=f'T_obs={T_obs_type:.1f}\np={p_val_type:.4f}')
axes[2].axvline(np.percentile(T_null_type, 95), color='orange', lw=1.5, ls='--', label='P95')
axes[2].set_xlabel('T')
axes[2].set_ylabel('Densidad')
axes[2].set_title('Fig 6c — Test ordenado')
axes[2].legend(fontsize=8)

plt.suptitle(r'Aplicación A: $H_0$ igualdad vs $H_A$ ordenada', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


### 3.2 Post-hoc Aplicación A — Permutación/Bootstrap rank-based para alternativas ordenadas

Tras rechazar $H_0$ global, identificamos qué saltos del orden son significativos usando el mismo principio de la hoja:

1. Se juntan los dos grupos del par.
2. Se calculan rangos conjuntos y rangos medios $R_i$, $R_j$.
3. Para el par ordenado esperado $m_i \leq m_j$, el estadístico es:

$$T_{ij}=R_j-R_i$$

4. Bajo $H_0: m_i=m_j$, las etiquetas son intercambiables; por tanto se remuestrea/permutan etiquetas para obtener la distribución nula de $T_{ij}$.

Con $k=4$ hay $\binom{4}{2}=6$ comparaciones. Aplicamos Bonferroni: $\alpha_{corr}=0.05/6=0.0083$.


In [ ]:
alpha_bonf_A = 0.05 / 6
pairs_A = list(combinations(range(4), 2))

posthoc_rows = []
posthoc_nulls = {}
for (i, j) in pairs_A:
    gi, gj = GROUP_ORDER[i], GROUP_ORDER[j]
    arr_i, arr_j = groups_type[i], groups_type[j]
    T_pair, T_pair_null, p_pair = permutation_test(
        [arr_i, arr_j], pair_order_statistic, n_perm=10_000,
        seed=100 + 10*i + j, alternative='greater'
    )
    posthoc_nulls[(i, j)] = T_pair_null
    diff_mean = arr_j.mean() - arr_i.mean()
    sig_bonf = p_pair < alpha_bonf_A
    posthoc_rows.append({
        'Par ordenado': f'{gi} ≤ {gj}',
        'n_i': len(arr_i), 'n_j': len(arr_j),
        'Media_i': round(arr_i.mean(), 3), 'Media_j': round(arr_j.mean(), 3),
        'Δmedia (j−i)': round(diff_mean, 3),
        'T_ij=Rj−Ri': round(T_pair, 2),
        'p_perm': round(p_pair, 5),
        'p_Bonf': round(min(p_pair * len(pairs_A), 1.0), 5),
        'Sig Bonf': 'sí' if sig_bonf else 'no',
    })

tab_posthoc = pd.DataFrame(posthoc_rows)
print(f'=== Post-hoc ordenado por permutación/bootstrap de etiquetas (Bonferroni α_corr={alpha_bonf_A:.4f}) ===')
print(tab_posthoc.to_string(index=False))
print()
print('Interpretación: p_perm pequeño indica evidencia de que el segundo grupo del par tiene rangos/tiempos mayores que el primero.')
print('Pares significativos tras Bonferroni:')
for _, row in tab_posthoc[tab_posthoc['Sig Bonf'] == 'sí'].iterrows():
    print(f'  {row["Par ordenado"]}: T={row["T_ij=Rj−Ri"]:.2f}, p_Bonf={row["p_Bonf"]:.5f}')


In [ ]:
# Fig 7 — Post-hoc ordenado: p-valores y distribuciones nulas pairwise
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

k = 4
p_mat = np.full((k, k), np.nan)
for idx, row in tab_posthoc.iterrows():
    i, j = pairs_A[idx]
    p_mat[i, j] = row['p_perm']

im = axes[0].imshow(np.nan_to_num(p_mat, nan=1.0), vmin=0, vmax=0.1, cmap='RdYlGn_r', aspect='auto')
plt.colorbar(im, ax=axes[0], label='p-valor permutación ordenada')
lbl = [g.replace('\n', ' ') for g in [GROUP_LABELS[g] for g in GROUP_ORDER]]
axes[0].set_xticks(range(k)); axes[0].set_xticklabels(lbl, fontsize=8, rotation=20, ha='right')
axes[0].set_yticks(range(k)); axes[0].set_yticklabels(lbl, fontsize=8)
axes[0].set_title('Fig 7a — Post-hoc ordenado: p_perm para i ≤ j')
for i in range(k):
    for j in range(k):
        if i >= j:
            txt = '—'
        else:
            val = p_mat[i, j]
            sig = '***' if val < alpha_bonf_A else ('*' if val < 0.05 else '')
            txt = f'{val:.4f}\n{sig}'
        axes[0].text(j, i, txt, ha='center', va='center', fontsize=8)

# Show null distributions for adjacent ordered contrasts, which define the proposed order.
adjacent_pairs = [(0, 1), (1, 2), (2, 3)]
colors_adj = ['tab:blue', 'tab:orange', 'tab:green']
for (i, j), color in zip(adjacent_pairs, colors_adj):
    null = posthoc_nulls[(i, j)]
    obs = tab_posthoc.iloc[pairs_A.index((i, j))]['T_ij=Rj−Ri']
    p_val = tab_posthoc.iloc[pairs_A.index((i, j))]['p_perm']
    axes[1].hist(null, bins=45, density=True, histtype='step', lw=1.8, color=color,
                 label=f'{GROUP_ORDER[i]}≤{GROUP_ORDER[j]} null')
    axes[1].axvline(obs, color=color, lw=2.2, ls='--', label=f'obs={obs:.1f}, p={p_val:.4f}')
axes[1].axvline(0, color='black', lw=1, alpha=0.5)
axes[1].set_xlabel('T_ij = R_j - R_i')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Fig 7b — Distribución nula en contrastes adyacentes')
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.25)

plt.suptitle('Post-hoc Aplicación A — test rank-based para alternativas ordenadas', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()



### 3.3 Aplicación B — Por línea de producción (k=3)

Usamos exactamente el mismo principio que para los tipos de cambio:

$$
H_0:m_{L17}=m_{L19}=m_{L14},
\qquad
H_A:m_{L17}\leq m_{L19}\leq m_{L14},
$$

con al menos una desigualdad estricta. Para tres grupos, el estadístico ordenado es

$$
T_3=(R_2-R_1)+(R_3-R_1)+(R_3-R_2)=2R_3-2R_1.
$$


In [ ]:

LINE_ORDER  = [17, 19, 14]
groups_line = [ch[ch['line']==l]['hours'].values for l in LINE_ORDER]
R_line = rank_means(groups_line)

print('=== Tabla 5 — Estadísticos por línea ===')
tab5 = pd.DataFrame([
    {'Línea':f'L{l}','n':len(arr),'Media (h)':round(arr.mean(),3),
     'Mediana (h)':round(np.median(arr),3),'Rango medio Rᵢ':round(ri,1)}
    for l,arr,ri in zip(LINE_ORDER, groups_line, R_line)
])
print(tab5.to_string(index=False))

T_obs_line, T_null_line, p_val_line = permutation_test(
    groups_line, ordered_rank_statistic, n_perm=10_000, seed=44, alternative='greater'
)
print(f'\nTest ordenado por permutación: T_obs={T_obs_line:.2f}, p={p_val_line:.4f}')
if p_val_line < 0.05:
    print('→ Rechazamos H₀: hay evidencia a favor del orden L17 ≤ L19 ≤ L14.')


In [ ]:

# Post-hoc líneas + Fig 8
alpha_bonf_B = 0.05 / 3
pairs_B = list(combinations(range(3), 2))
line_posthoc_rows = []
line_posthoc_nulls = {}

for (i, j) in pairs_B:
    li, lj = LINE_ORDER[i], LINE_ORDER[j]
    arr_i, arr_j = groups_line[i], groups_line[j]
    T_pair, T_pair_null, p_pair = permutation_test(
        [arr_i, arr_j], pair_order_statistic, n_perm=10_000,
        seed=300 + 10*i + j, alternative='greater'
    )
    line_posthoc_nulls[(i, j)] = T_pair_null
    sig_bonf = p_pair < alpha_bonf_B
    line_posthoc_rows.append({
        'Par ordenado': f'L{li} ≤ L{lj}',
        'n_i': len(arr_i), 'n_j': len(arr_j),
        'Media_i': round(arr_i.mean(), 3), 'Media_j': round(arr_j.mean(), 3),
        'Δmedia (j−i)': round(arr_j.mean() - arr_i.mean(), 3),
        'T_ij=Rj−Ri': round(T_pair, 2),
        'p_perm': round(p_pair, 5),
        'p_Bonf': round(min(p_pair * len(pairs_B), 1.0), 5),
        'Sig Bonf': 'sí' if sig_bonf else 'no',
    })

tab_line_posthoc = pd.DataFrame(line_posthoc_rows)
print(f'=== Post-hoc ordenado líneas por permutación (Bonferroni α_corr={alpha_bonf_B:.4f}) ===')
print(tab_line_posthoc.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
pal_l = sns.color_palette('Set2', 3)
bp_l = axes[0].boxplot(groups_line, patch_artist=True, medianprops={'color':'red','lw':2})
for patch, color in zip(bp_l['boxes'], pal_l):
    patch.set_facecolor(color)
axes[0].set_xticks(range(1,4))
axes[0].set_xticklabels([f'L{l}\n(n={len(ch[ch.line==l])})' for l in LINE_ORDER])
axes[0].set_ylabel('Horas')
axes[0].set_title('Fig 8a — Boxplots por línea')
axes[0].grid(axis='y', alpha=0.3)
for i, arr in enumerate(groups_line):
    axes[0].text(i+1, arr.mean()+0.08, f'μ={arr.mean():.2f}h',
                 ha='center', fontsize=9, fontweight='bold', color='navy')

axes[1].hist(T_null_line, bins=60, density=True, color='lightcoral', edgecolor='white', lw=0.4,
             label='Distribución nula')
axes[1].axvline(T_obs_line, color='darkred', lw=2.5, label=f'T_obs={T_obs_line:.1f} (p={p_val_line:.4f})')
axes[1].axvline(np.percentile(T_null_line,95), color='orange', lw=1.5, ls='--',
                label=f'P95={np.percentile(T_null_line,95):.1f}')
axes[1].set_xlabel('T')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Fig 8b — Distribución nula ordenada')
axes[1].legend(fontsize=9)

p_mat_l = np.full((3, 3), np.nan)
for idx, row in tab_line_posthoc.iterrows():
    i, j = pairs_B[idx]
    p_mat_l[i, j] = row['p_perm']
im = axes[2].imshow(np.nan_to_num(p_mat_l, nan=1.0), vmin=0, vmax=0.1, cmap='RdYlGn_r', aspect='auto')
plt.colorbar(im, ax=axes[2], label='p-valor permutación ordenada')
axes[2].set_xticks(range(3)); axes[2].set_xticklabels([f'L{l}' for l in LINE_ORDER])
axes[2].set_yticks(range(3)); axes[2].set_yticklabels([f'L{l}' for l in LINE_ORDER])
axes[2].set_title('Fig 8c — Post-hoc líneas')
for i in range(3):
    for j in range(3):
        if i >= j:
            txt = '—'
        else:
            val = p_mat_l[i, j]
            sig = '***' if val < alpha_bonf_B else ('*' if val < 0.05 else '')
            txt = f'{val:.4f}\n{sig}'
        axes[2].text(j, i, txt, ha='center', va='center', fontsize=9)

plt.suptitle(r'Aplicación B: $H_0$ igualdad vs $H_A$: $m_{L17}\leq m_{L19}\leq m_{L14}$',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 4. Síntesis y conclusiones

In [ ]:

print('=== Tabla 6 — Resumen de resultados ===')
tab6 = pd.DataFrame([
    {'Método':'GoF global','H₀/H_A':'familia seleccionada en 2.1 validada en H2',
     'Estadístico':f'AIC_H2={selected_global_aic_h2}', 'Decisión':f'{selected_global_name} (KS_p={selected_global_ks_p})'},
    {'Método':'T ordenado tipos (k=4)','H₀/H_A':'C_pack≤C_brand≤C0≤C_envase',
     'Estadístico':f'T={T_obs_type:.1f}','Decisión':f'Rechazar H₀ (p={p_val_type:.4f})'},
    {'Método':'T ordenado líneas (k=3)','H₀/H_A':'L17≤L19≤L14',
     'Estadístico':f'T={T_obs_line:.1f}','Decisión':f'Rechazar H₀ (p={p_val_line:.4f})'},
])
print(tab6.to_string(index=False))


In [ ]:
print('=== IMPLICACIONES PARA EL OPTIMIZADOR ===')
print()
print(f'BCN: {ch["node"].nunique()} nodos  |  {len(edge_priors)} aristas')
print(f'  {n_specific} priors específicos  |  {n_global} usan prior global Gamma({a_g:.3f},{sc_g:.3f})')
print()
print('ORDEN DE CHANGEOVERS (post-hoc ordenado por permutación + Bonferroni):')
for g, arr in zip(GROUP_ORDER, groups_type):
    print(f'  {g:<12} μ={arr.mean():.2f}h')
print('  → C_envase (referencia) > C0_self > C_brand > C_pack')
print('  → Cambiar de referencia de etiqueta es MÁS COSTOSO que cambiar de marca.')
print()
print('LÍNEAS: L14 significativamente más lenta (p<0.001 post-hoc L14>L17, L14>L19)')
print('  → Asignar a L14 secuencias con pocos C_envase consecutivos.')
